Step 1


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from xgboost import XGBClassifier
from fuzzywuzzy import process
import joblib

# Step 2: Load and Explore Data
# Load Datasets

In [2]:
ipo = pd.read_csv("/Users/adi/Desktop/Mission/IPO Allotment Prediction model/ipo.csv", header=1)
cleaned_ipo = pd.read_csv("/Users/adi/Desktop/Mission/IPO Allotment Prediction model/cleaned_ipo_data 2022-25.csv")


In [3]:
# Display First Few Rows
print(ipo.head())
print(cleaned_ipo.head(380))

   Unnamed: 0        Date         IPO Name  Issue Size  (in crores)     QIB  \
0           0  17-10-2022  Electronics Mar                   500.00   58.81   
1           1  26-09-2022  Harsha Engineer                   755.00  113.82   
2           2  15-09-2022              TMB                   792.00    0.51   
3           3  06-09-2022  Dreamfolks Serv                   562.10   27.48   
4           4  26-08-2022        Syrma SGS                   840.13   42.42   

     HNI    RII  Total  Issue  Listing Open  Listing Close  Listing Gains(%)  \
0  15.39   8.27  24.23     59         85.90          84.45             43.14   
1  40.36  12.44  47.19    330        495.00         485.90             47.24   
2   1.77   3.44   1.39    525        480.05         508.45             -3.15   
3  14.18  24.19  23.25    326        455.00         462.65             41.92   
4   7.13   2.84  15.59    220        262.00         313.05             42.30   

      CMP  Current  Gains (%)  
0   91.90   

In [6]:
# Check for Missing Values
print(ipo.isnull().sum())
print(cleaned_ipo.isnull().sum())

Unnamed: 0                 0
Date                       0
IPO Name                   0
Issue Size  (in crores)    0
QIB                        0
HNI                        0
RII                        0
Total                      0
Issue                      0
Listing Open               0
Listing Close              0
Listing Gains(%)           0
CMP                        0
Current  Gains (%)         2
dtype: int64
Name             0
Listing_Date     0
Issue_Price      0
Listing_Price    0
LTP              1
Returns          2
dtype: int64


In [10]:
#combine datasets
## Fuzzy Match Company Names
# Fuzzy Match Company Names
def fuzzy_merge(df1, df2, key1, key2, threshold=80):
    matches = []
    for name in df1[key1]:
        result = process.extractOne(name, df2[key2])
        if result:
            match, score = result[0], result[1]
            if score >= threshold:
                matches.append(match)
            else:
                matches.append(None)
        else:
            matches.append(None)
    df1['matched_name'] = matches
    return df1.merge(df2, left_on='matched_name', right_on=key2, how='inner')

combined = fuzzy_merge(ipo, cleaned_ipo, 'IPO Name', 'Name')

In [11]:
# Display Combined Dataset
print(combined.head())

   Unnamed: 0        Date         IPO Name  Issue Size  (in crores)     QIB  \
0           0  17-10-2022  Electronics Mar                   500.00   58.81   
1           1  26-09-2022  Harsha Engineer                   755.00  113.82   
2           3  06-09-2022  Dreamfolks Serv                   562.10   27.48   
3           4  26-08-2022        Syrma SGS                   840.13   42.42   
4           8  24-05-2022      Venus Pipes                   165.42    6.01   

     HNI    RII  Total  Issue  Listing Open  ...  Listing Gains(%)     CMP  \
0  15.39   8.27  24.23     59          85.9  ...             43.14   91.90   
1  40.36  12.44  47.19    330         495.0  ...             47.24  415.00   
2  14.18  24.19  23.25    326         455.0  ...             41.92  417.50   
3   7.13   2.84  15.59    220         262.0  ...             42.30  276.45   
4   6.59  10.90   8.58    326         352.0  ...              7.90  754.50   

   Current  Gains (%)                        matched_nam

In [12]:
# preprocess the data.
# Target Variable (Binary)
combined['Target'] = ((combined['Listing Gains(%)'].fillna(0) > 0) | (combined['Returns'].fillna(0) > 0)).astype(int)


In [13]:
# Log Transform Skewed Features
# The column 'Issue Size  (in crores)' has a space at the end.
combined['Log_Issue_Size'] = np.log1p(combined['Issue Size  (in crores)'])


In [14]:
# Investor Category (Categorical)
combined['Investor_Category'] = pd.cut(
    combined['RII'], 
    bins=[-np.inf, 50, 100, np.inf], 
    labels=['Retail', 'HNI', 'Institutional']
)


In [15]:
# Drop Unnecessary Columns
combined = combined[['Log_Issue_Size', 'Total', 'Investor_Category', 'Target']]


# Step 5: Train-Test Split
# Define Features and Target

In [16]:
X = combined.drop(columns=['Target'])
y = combined['Target']


In [17]:

# Calculate scale_pos_weight for handling class imbalance
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

In [18]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Step 6: Build Pipeline

In [19]:
# Preprocessing Pipelines
numerical_features = ['Log_Issue_Size', 'Total']
categorical_features = ['Investor_Category']

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numerical_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

# Full Pipeline with Model
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, reg_alpha=0.1))
])


# Step 7: Hyperparameter Tuning

In [20]:
# Hyperparameter Grid
param_grid = {
    'classifier__n_estimators': [50, 100, 150],
    'classifier__max_depth': [3, 4, 5],
    'classifier__learning_rate': [0.05, 0.1, 0.15],
    'classifier__gamma': [0, 0.1, 0.2]
}

# Grid Search
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc')
grid_search.fit(X_train, y_train)

# Best Parameters
print("Best Parameters:", grid_search.best_params_)



Best Parameters: {'classifier__gamma': 0, 'classifier__learning_rate': 0.05, 'classifier__max_depth': 3, 'classifier__n_estimators': 50}


Step 8: Evaluate the Model

# Step 8: Evaluate the Model
# Predict Probabilities
y_pred_proba = grid_search.predict_proba(X_test)[:, 1]

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC-AUC: {roc_auc:.2f}")

# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
plt.legend()
plt.show()

# Classification Report
y_pred = grid_search.predict(X_test)
print(classification_report(y_test, y_pred))


In [22]:
# Step 9: Deploy the Model

In [23]:
# Save Model
joblib.dump(grid_search, "/Users/adi/Desktop/Mission/IPO Allotment Prediction model/ipo_model.pkl")


['/Users/adi/Desktop/Mission/IPO Allotment Prediction model/ipo_model.pkl']